# CS311 Sudoku Solver
Run everything with **Runtime → Run all**. The last cell prints the results.

In [3]:
%%writefile solver_basic.py
# solver_basic.py
# Basic backtracking Sudoku solver.
# Picks the NEXT empty cell in row-major order (left to right, top to bottom).
# Search node = one call to the recursive function bt().

def solve_basic(puzzle):
    grid = [row[:] for row in puzzle]          # copy so the original clues are never changed

    # Sets of values already used in each row, column, and 3x3 box
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empties = []                                # list of (row, col) cells that are 0

    for r in range(9):
        for c in range(9):
            v = grid[r][c]
            if v == 0:
                empties.append((r, c))
            else:
                rows[r].add(v)
                cols[c].add(v)
                boxes[(r // 3) * 3 + c // 3].add(v)

    nodes = 0

    def bt(k):                                  # k = index of the next empty cell to fill
        nonlocal nodes
        nodes += 1                              # count this call as one search node
        if k == len(empties):
            return True                         # all empty cells are filled -> solved
        r, c = empties[k]
        b = (r // 3) * 3 + c // 3
        for v in range(1, 10):
            if v not in rows[r] and v not in cols[c] and v not in boxes[b]:
                grid[r][c] = v                  # place
                rows[r].add(v); cols[c].add(v); boxes[b].add(v)
                if bt(k + 1):
                    return True
                grid[r][c] = 0                  # undo (backtrack)
                rows[r].remove(v); cols[c].remove(v); boxes[b].remove(v)
        return False                            # no value worked -> dead end

    solved = bt(0)
    return (grid if solved else None), nodes


Overwriting solver_basic.py


In [4]:
%%writefile solver_mrv.py
# solver_mrv.py
# Backtracking Sudoku solver with the Minimum Remaining Values (MRV) heuristic.
# On every call it picks the empty cell with the FEWEST legal values.
# Search node = one call to the recursive function bt()  (same definition as basic).

def solve_mrv(puzzle):
    grid = [row[:] for row in puzzle]          # copy so the original clues are never changed

    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empties = []

    for r in range(9):
        for c in range(9):
            v = grid[r][c]
            if v == 0:
                empties.append((r, c))
            else:
                rows[r].add(v)
                cols[c].add(v)
                boxes[(r // 3) * 3 + c // 3].add(v)

    nodes = 0

    def bt():
        nonlocal nodes
        nodes += 1                              # count this call as one search node

        # MRV step: scan every still-empty cell and keep the one with fewest candidates
        best, best_cands = None, None
        for (r, c) in empties:
            if grid[r][c] != 0:
                continue                        # already filled in this branch
            b = (r // 3) * 3 + c // 3
            cands = [v for v in range(1, 10)
                     if v not in rows[r] and v not in cols[c] and v not in boxes[b]]
            if best is None or len(cands) < len(best_cands):
                best, best_cands = (r, c), cands
                if len(cands) == 0:
                    break                       # a cell with 0 options -> this branch is dead

        if best is None:
            return True                         # no empty cells left -> solved

        r, c = best
        b = (r // 3) * 3 + c // 3
        for v in best_cands:                    # if best_cands is empty, loop is skipped -> False
            grid[r][c] = v                      # place
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
            if bt():
                return True
            grid[r][c] = 0                      # undo (backtrack)
            rows[r].remove(v); cols[c].remove(v); boxes[b].remove(v)
        return False

    solved = bt()
    return (grid if solved else None), nodes


Overwriting solver_mrv.py


In [5]:
%%writefile check.py
# check.py
# Quick test for the Nov 5 checkpoint: runs both solvers on the same puzzles.

from solver_basic import solve_basic
from solver_mrv import solve_mrv

puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]


def is_valid_solution(sol, original):
    full = set(range(1, 10))
    for i in range(9):
        if set(sol[i]) != full:                                   # row
            return False
        if set(sol[r][i] for r in range(9)) != full:              # column
            return False
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            box = {sol[r][c] for r in range(br, br + 3) for c in range(bc, bc + 3)}
            if box != full:
                return False
    for r in range(9):                                            # clues unchanged
        for c in range(9):
            if original[r][c] != 0 and original[r][c] != sol[r][c]:
                return False
    return True


def show(grid):
    for row in grid:
        print(" ".join(str(v) for v in row))


completed, _ = solve_basic(puzzle)          # a finished board for the 1-node test

tests = [("Sample puzzle", puzzle), ("Completed board", completed)]

for name, p in tests:
    print("=====", name, "=====")
    sol_b, nodes_b = solve_basic(p)
    sol_m, nodes_m = solve_mrv(p)
    print("Basic: nodes =", nodes_b, "| valid =", is_valid_solution(sol_b, p))
    print("MRV:   nodes =", nodes_m, "| valid =", is_valid_solution(sol_m, p))
    print("Same solution:", sol_b == sol_m)
    print()

print("Solved grid:")
show(solve_mrv(puzzle)[0])


Writing check.py


In [6]:
!python check.py

===== Sample puzzle =====
Basic: nodes = 4209 | valid = True
MRV:   nodes = 52 | valid = True
Same solution: True

===== Completed board =====
Basic: nodes = 1 | valid = True
MRV:   nodes = 1 | valid = True
Same solution: True

Solved grid:
5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9
